In [ ]:
from pathlib import Path

def _sionna_repo_root():
    """Find project root when Jupyter is started from root or notebooks/."""
    for candidate in (Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent):
        if (candidate / "scenes" / "University.xml").is_file():
            return candidate
    raise FileNotFoundError("Run Jupyter from the repository root or notebooks/ directory")


import numpy as np
import sionna.rt
from sionna.rt import load_scene, PlanarArray, Transmitter, Receiver, PathSolver, Camera

from PIL import Image
from IPython.display import display

# ============================================================
# Sionna RT: 3D scene + ray paths
# - preview(): interactive viewer (camera control by mouse)
# - render_to_file(): save a zoomed snapshot with a fixed Camera
# ============================================================

# 1) Load a built-in 3D scene (fresh scene each run)
scene = load_scene(sionna.rt.scene.munich, merge_shapes=True)
scene.frequency = 28e9  # 28 GHz

# 2) Define antenna arrays (IMPORTANT: keyword-only arguments)
scene.tx_array = PlanarArray(
    num_rows=1,
    num_cols=1,
    vertical_spacing=0.5,
    horizontal_spacing=0.5,
    pattern="tr38901",
    polarization="V"
)

scene.rx_array = PlanarArray(
    num_rows=1,
    num_cols=1,
    vertical_spacing=0.5,
    horizontal_spacing=0.5,
    pattern="dipole",
    polarization="V"
)

# 3) Add UE and BS (make them BIG so you can see them)
tx_pos = np.array([8.5, 21.0, 1.5])
rx_pos = np.array([45.0, 90.0, 10.0])

tx = Transmitter(name="UE", position=tx_pos.tolist(), display_radius=35)
rx = Receiver(name="BS", position=rx_pos.tolist(), display_radius=35)

scene.add(tx)
scene.add(rx)

# Orient UE toward BS (affects antenna orientation)
tx.look_at(rx)

# 4) Ray tracing (keep it simple to avoid clutter)
solver = PathSolver()
paths = solver(
    scene=scene,
    max_depth=2,                 # reflections depth
    los=True,
    specular_reflection=True,
    diffuse_reflection=False,
    refraction=True,
    synthetic_array=False,
    seed=1
)

# 5) Interactive 3D preview (no camera kwargs in your version)
# Use mouse to zoom/pan/rotate; UE/BS are now much bigger
scene.preview(paths=paths, clip_at=150)

# 6) Save a "zoomed" snapshot with a fixed Camera (this DOES support Camera)
mid = 0.5 * (tx_pos + rx_pos)
cam_pos = mid + np.array([-80.0, -80.0, 60.0])  # move closer by reducing magnitudes

camera = Camera(
    position=cam_pos.tolist(),
    look_at=mid.tolist()
)

out_file = str(_sionna_repo_root() / "figures" / "rt_paths_zoom.png")
scene.render_to_file(
    camera=camera,
    paths=paths,
    filename=out_file,
    resolution=[1100, 800],
    num_samples=128,
    clip_at=150
)

print(f"Saved snapshot: {out_file}")

# Display the saved image inside the notebook
display(Image.open(out_file))
